In [ ]:
import os
import pandas as pd
import numpy as np
import pickle

# Setup paths
RAW_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("Loading ratings data...")
# Read columns with safe datatypes first (treating timestamp as an object string)
dtypes = {'userId': np.int32, 'movieId': np.int32, 'rating': np.float32, 'timestamp': str}
df = pd.read_csv(os.path.join(RAW_DIR, 'rating.csv'), dtype=dtypes)

print(f"Total raw interactions loaded: {len(df):,}")

# Convert the string date into an optimized Pandas datetime object
print("Parsing date strings into datetime objects...")
df['timestamp'] = pd.to_datetime(df['timestamp'])

# 1. Implicit Conversion: Keep interactions where rating >= 3.5
df = df[df['rating'] >= 3.5].copy()
df.drop(columns=['rating'], inplace=True)
print(f"Interactions after implicit conversion (rating >= 3.5): {len(df):,}")

# 2. Filter out noisy users with less than 5 interactions to ensure robust training sequences
user_counts = df['userId'].value_counts()
df = df[df['userId'].isin(user_counts[user_counts >= 5].index)]
print(f"Interactions after removing users with < 5 watches: {len(df):,}")

# 3. Sort chronologically per user using the datetime object
print("Sorting data chronologically...")
df.sort_values(by=['userId', 'timestamp'], ascending=[True, True], inplace=True)
df.reset_index(drop=True, inplace=True)
print("Data sorted successfully.")

Loading ratings data...
Total raw interactions loaded: 20,000,263
Parsing date strings into datetime objects...
Interactions after implicit conversion (rating >= 3.5): 12,195,566
Interactions after removing users with < 5 watches: 12,192,944
Sorting data chronologically...
Data sorted successfully.


In [2]:
df.head(5)

,userId,movieId,timestamp
0,1,924,2004-09-10 03:06:38
1,1,919,2004-09-10 03:07:01
2,1,2683,2004-09-10 03:07:30
3,1,1584,2004-09-10 03:07:36
4,1,1079,2004-09-10 03:07:45


In [3]:

# Create mapping dictionaries
unique_users = df['userId'].unique()
unique_movies = df['movieId'].unique()

user_to_idx = {uid: int(idx + 1) for idx, uid in enumerate(unique_users)}
movie_to_idx = {mid: int(idx + 1) for idx, mid in enumerate(unique_movies)}

# Apply the token transformations
df['user_idx'] = df['userId'].map(user_to_idx).astype(np.int32)
df['movie_idx'] = df['movieId'].map(movie_to_idx).astype(np.int32)

# Save the mappings (Crucial for the Phase 4 FastAPI server to translate inputs)
with open(os.path.join(PROCESSED_DIR, 'mappings.pkl'), 'wb') as f:
    pickle.dump({'user_to_idx': user_to_idx, 'movie_to_idx': movie_to_idx}, f)

print(f"Unique Users mapped: {len(user_to_idx)}")
print(f"Unique Movies mapped: {len(movie_to_idx)}")

Unique Users mapped: 137477
Unique Movies mapped: 22884


In [4]:
MAX_SEQ_LEN = 50  # Hyperparameter T for SASRec and BSARec

print("Generating sequences...")
user_grouped = df.groupby('user_idx')['movie_idx'].apply(list).to_dict()
print(f"Total users with sequences: {len(user_grouped):,}")

train_sequences = []
val_targets = []
test_targets = []

for u_idx, seq in user_grouped.items():
    if len(seq) < 3:
        continue # Skipped out via pipeline routing constraints
        
    # Split targets chronologically
    test_target = seq[-1]
    val_target = seq[-2]
    train_seq = seq[:-2]
    
    # Process training sequence with padding or truncation
    if len(train_seq) > MAX_SEQ_LEN:
        train_seq = train_seq[-MAX_SEQ_LEN:]
    else:
        padding_len = MAX_SEQ_LEN - len(train_seq)
        train_seq = [0] * padding_len + train_seq
        
    train_sequences.append([u_idx] + train_seq)
    val_targets.append([u_idx, val_target])
    test_targets.append([u_idx, test_target])

# Convert to arrays and export
np.save(os.path.join(PROCESSED_DIR, 'train_seqs.npy'), np.array(train_sequences, dtype=np.int32))
np.save(os.path.join(PROCESSED_DIR, 'val_targets.npy'), np.array(val_targets, dtype=np.int32))
np.save(os.path.join(PROCESSED_DIR, 'test_targets.npy'), np.array(test_targets, dtype=np.int32))

print(f"Saved {len(train_sequences):,} processed training sequences.")

Generating sequences...
Total users with sequences: 137,477
Saved 137,477 processed training sequences.


In [5]:
print("Processing tag genome scores...")
genome_df = pd.read_csv(os.path.join(RAW_DIR, 'genome_scores.csv'))

# Pivot table into an Item x Tag relevance matrix
genome_pivot = genome_df.pivot(index='movieId', columns='tagId', values='relevance').fillna(0.0)

# Re-align rows to match our custom token mapping index order
num_movies = len(movie_to_idx)
num_tags = genome_pivot.shape[1]
dense_genome_matrix = np.zeros((num_movies + 1, num_tags), dtype=np.float32) # row 0 remains padded zeros

for old_mid, new_idx in movie_to_idx.items():
    if old_mid in genome_pivot.index:
        dense_genome_matrix[new_idx] = genome_pivot.loc[old_mid].values

np.save(os.path.join(PROCESSED_DIR, 'genome_matrix.npy'), dense_genome_matrix)
print(f"Exported dense genome feature matrix with shape: {dense_genome_matrix.shape}")

Processing tag genome scores...
Exported dense genome feature matrix with shape: (22885, 1128)


In [6]:
dense_genome_matrix

array([[0.     , 0.     , 0.     , ..., 0.     , 0.     , 0.     ],
       [0.0235 , 0.01825, 0.1065 , ..., 0.01375, 0.057  , 0.01875],
       [0.01775, 0.01725, 0.1205 , ..., 0.03825, 0.07275, 0.021  ],
       ...,
       [0.     , 0.     , 0.     , ..., 0.     , 0.     , 0.     ],
       [0.     , 0.     , 0.     , ..., 0.     , 0.     , 0.     ],
       [0.     , 0.     , 0.     , ..., 0.     , 0.     , 0.     ]],
      shape=(22885, 1128), dtype=float32)

In [7]:
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import csr_matrix

print("Constructing sparse interaction matrix for Baseline...")
# Build standard interaction sparse matrix
rows = df['user_idx'].values
cols = df['movie_idx'].values
data = np.ones(len(df), dtype=np.float32)

interaction_matrix = csr_matrix((data, (rows, cols)), shape=(len(user_to_idx) + 1, len(movie_to_idx) + 1))

print("Training Truncated SVD model...")
svd = TruncatedSVD(n_components=32, random_state=42)
user_embeddings = svd.fit_transform(interaction_matrix)
item_embeddings = svd.components_.T

print("Computing baseline scores for test suite verification...")
# Reconstruct top scores for sample validation
test_arr = np.load(os.path.join(PROCESSED_DIR, 'test_targets.npy'))

# Evaluate simple Hit Ratio at 10 for a validation sample
hits = 0
sample_size = min(5000, len(test_arr)) # Sample to check speeds quickly

for i in range(sample_size):
    u, target = test_arr[i]
    user_vec = user_embeddings[u]
    scores = np.dot(item_embeddings, user_vec)
    top_k_idx = np.argsort(scores)[-10:] # Top 10 predictions
    if target in top_k_idx:
        hits += 1

hr_10 = hits / sample_size
print(f"Experiment 1 Baseline Control Complete -> Sample Hit Ratio@10: {hr_10:.4f}")

Constructing sparse interaction matrix for Baseline...
Training Truncated SVD model...
Computing baseline scores for test suite verification...
Experiment 1 Baseline Control Complete -> Sample Hit Ratio@10: 0.1008
